In [0]:
print(spark)
print(spark.version)

In [0]:
df = [(1, "Alice"),
    (2, "Bob"),
    (3, "Charlie")]

df = spark.createDataFrame(df,["id","name"])
df2 = df.filter(df.id>1)
print("DataFrame created and filtered")

In [0]:
df2.display()

In [0]:
df.explain("formatted")

In [0]:
display(dbutils.fs.ls("abfss://raw@fintechdllasya.dfs.core.windows.net/customers/"))

In [0]:
display(dbutils.fs.ls("abfss://raw@fintechdllasya.dfs.core.windows.net/customers/2026-08-17/"))

In [0]:
customers_df = spark.read.option("header","true")\
                          .option("inferSchema","true")\
                          .csv("abfss://raw@fintechdllasya.dfs.core.windows.net/customers/2026-08-17/")
display(customers_df)

In [0]:
customers_df.printSchema()

In [0]:
from pyspark.sql import functions as F

customers_df.select(
    F.count("*").alias("total_rows"),
    F.countDistinct("customer_id").alias("unique_customers"),
    F.sum(F.col("customer_id").isNull().cast("int")).alias("null_customer_ids"),
    F.sum(F.col("email").isNull().cast("int")).alias("null_emails"),
    F.sum(F.col("phone").isNull().cast("int")).alias("null_phones"),
    F.sum(F.col("country").isNull().cast("int")).alias("null_countries"),
    F.sum(F.col("customer_type").isNull().cast("int")).alias("null_customer_types")
).display()

In [0]:
customers_df.groupBy("customer_type").count().display()

In [0]:
customers_df.groupBy("country").count().display()

In [0]:
customers_18_df = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv(
             "abfss://raw@fintechdllasya.dfs.core.windows.net/"
             "customers/2026-08-18/customers.csv"
         )
)

In [0]:
print("Day 17:", customers_df.count())
print("Day 18:", customers_18_df.count())

In [0]:
customers_df.select("customer_id").distinct().join(
    customers_18_df.select("customer_id").distinct(),
    "customer_id",
    "inner"
).count()

In [0]:
from pyspark.sql import functions as F

changed_customers = (
    customers_df.alias("d17")
    .join(
        customers_18_df.alias("d18"),
        "customer_id",
        "inner"
    )
    .filter(
        (F.col("d17.name") != F.col("d18.name")) |
        (F.col("d17.email") != F.col("d18.email")) |
        (F.col("d17.phone") != F.col("d18.phone")) |
        (F.col("d17.country") != F.col("d18.country")) |
        (F.col("d17.customer_type") != F.col("d18.customer_type"))
    )
)

print("Changed customers:", changed_customers.count())

In [0]:
display(
    dbutils.fs.ls(
        "abfss://raw@fintechdllasya.dfs.core.windows.net/"
    )
)

In [0]:
display(
    dbutils.fs.ls(
        "abfss://raw@fintechdllasya.dfs.core.windows.net/adf_test/"
    )
)